# Broadcast Study Analysis

This notebook imports data collected by a Daphne parameter study, analyzes
scalability and performance metrics of broadcasts, and visualizes results.

## 1. Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dask.dataframe as dd
%matplotlib inline

# Enable copy-on-write to follow pandas recommendation.
# see https://pandas.pydata.org/docs/user_guide/copy_on_write.html
pd.options.mode.copy_on_write = True

In [ ]:
# Load collected data from Parquet file.
# Replace '../data.parquet' with the path to your data file.
# As a reminder, please run `go run ./daphne study broadcast` to produce this 
# file. Data is loaded in chunks and processed to categorize columns to limit 
# peak memory usage.
delayedData = dd.read_parquet(
    '../../data.parquet',
    columns=['timestamp', 'mark', 'from', 'to', 'rid', 'NumNodes', 'Broadcast']
)

def process_chunk(chunk):
    chunk = dd.from_delayed(chunk).compute()
    for col in ['mark', 'from', 'to', 'Broadcast']:
        chunk[col] = chunk[col].astype('category')
    return chunk

data = pd.concat((process_chunk(chunk) for chunk in delayedData.to_delayed()), ignore_index=True)

## 2. Number of Messages
The following chart illustrates the number of messages being transferred over
the network depending on the number of involved nodes and broadcast protocols.


In [ ]:
# Filter message sent and study started rows
msgsent = data[data['mark'] == 'MsgSent']
scenarios = data[data['mark'] == 'StudyStarted'][['rid', 'NumNodes', 'Broadcast']]

# Count messages per experiment run
msg_counts = msgsent.groupby('rid').size().reset_index(name='MsgCount')

# Join with experiment parameters
merged = pd.merge(msg_counts, scenarios, on='rid')

# Group by NumNodes and Broadcast, calculate mean and std
stats = merged.groupby(['NumNodes', 'Broadcast'])['MsgCount'].agg(['mean', 'std', 'count']).reset_index()

# Pivot for grouped bar plot
pivot = stats.pivot(index='NumNodes', columns='Broadcast', values='mean')
pivot_std = stats.pivot(index='NumNodes', columns='Broadcast', values='std')

# Get number of runs per group
pivot_counts = stats.pivot(index='NumNodes', columns='Broadcast', values='count')
min_counts = pivot_counts.min().min()
max_counts = pivot_counts.max().max()
num_runs_text = f'runs per group: min={min_counts}, max={max_counts}'

# Plot grouped bar chart with error bars
pivot.plot(kind='bar', yerr=pivot_std, figsize=(10, 6), capsize=4)
plt.title('Messages Transferred by NumNodes and Broadcast Protocol (' + num_runs_text + ')')
plt.xlabel('NumNodes')
plt.ylabel('Mean Number of Messages Transferred')
plt.legend(title='Broadcast Protocol')
plt.grid(True)
plt.tight_layout()
plt.show()

The following illustrates the distributions of number of messages sent by individual nodes.

In [ ]:
# Filter message sent and study started rows
msgsent = data[data['mark'] == 'MsgSent']
scenarios = data[data['mark'] == 'StudyStarted'][['rid', 'NumNodes', 'Broadcast']]

# Count messages per experiment run
msg_counts = msgsent.groupby('rid').size().reset_index(name='MsgCount')

# Join with experiment parameters
merged = pd.merge(msg_counts, scenarios, on='rid')

# Normalize message count by number of nodes
merged['MsgCount'] = merged['MsgCount'] / merged['NumNodes']

# Group by NumNodes and Broadcast, calculate mean and std
stats = merged.groupby(['NumNodes', 'Broadcast'], observed=True)['MsgCount'].agg(['mean', 'std', 'count']).reset_index()

# Pivot for grouped bar plot
pivot = stats.pivot(index='NumNodes', columns='Broadcast', values='mean')
pivot_std = stats.pivot(index='NumNodes', columns='Broadcast', values='std')

# Get number of runs per group
pivot_counts = stats.pivot(index='NumNodes', columns='Broadcast', values='count')
min_counts = pivot_counts.min().min()
max_counts = pivot_counts.max().max()
num_runs_text = f'runs per group: min={min_counts}, max={max_counts}'

# Plot grouped bar chart with error bars
pivot.plot(kind='bar', yerr=pivot_std, figsize=(10, 6), capsize=4)
plt.title('Mean Messages Sent per Node by NumNodes and Broadcast Protocol (' + num_runs_text + ')')
plt.xlabel('NumNodes')
plt.ylabel('Mean Number of Messages Sent per Node')
plt.legend(title='Broadcast Protocol')
plt.grid(True)
plt.tight_layout()
plt.show()

The following chart is a refinement of the per-node message-sent chart providing a break-down on the
number of messages send per node.

In [ ]:
# Get all MsgSent entries
msg_sent = data[data['mark'] == 'MsgSent']

# Get some statistics on the number of sent messages per node
msg_counts = msg_sent.groupby(['rid', 'from'], observed=True).size().reset_index(name='MsgSentCount')

# Merge with run metadata (NumNodes, Broadcast)
msg_counts = msg_counts.merge(scenarios, on='rid', how='left')


# Get number of runs per group
pivot_counts = stats.pivot(index='NumNodes', columns='Broadcast', values='count')
min_counts = pivot_counts.min().min()
max_counts = pivot_counts.max().max()
num_runs_text = f'runs per group: min={min_counts}, max={max_counts}'

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=msg_counts,
    x='NumNodes',
    y='MsgSentCount',
    hue='Broadcast',
)
plt.title('Distribution of Sent Messages per Node by Broadcast Protocol and NumNodes (' + num_runs_text + ')')
plt.xlabel('Number of Nodes')
plt.ylabel('Sent Messages per Node')
plt.legend(title='Broadcast Protocol')
plt.tight_layout()
plt.show()


In [ ]:
# Get all MsgReceived entries
msg_received = data[data['mark'] == 'MsgReceived']

# Get some statistics on the number of received messages per node
msg_counts = msg_received.groupby(['rid', 'to'], observed=True).size().reset_index(name='MsgReceivedCount')

# Merge with run metadata (NumNodes, Broadcast)
msg_counts = msg_counts.merge(scenarios, on='rid', how='left')


# Get number of runs per group
pivot_counts = stats.pivot(index='NumNodes', columns='Broadcast', values='count')
min_counts = pivot_counts.min().min()
max_counts = pivot_counts.max().max()
num_runs_text = f'runs per group: min={min_counts}, max={max_counts}'

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=msg_counts,
    x='NumNodes',
    y='MsgReceivedCount',
    hue='Broadcast',
)
plt.title('Distribution of Received Messages per Node by Broadcast Protocol and NumNodes (' + num_runs_text + ')')
plt.xlabel('Number of Nodes')
plt.ylabel('Received Messages per Node')
plt.legend(title='Broadcast Protocol')
plt.tight_layout()
plt.show()


Todo for follow-up work:
 - Plot broadcast latency (time from begin of broadcast to 90%, 95%, 99% reach)